In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw3.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 3: Preprocessing 
### Associated lectures: [Lectures 4, 5, 6](https://github.com/UBC-CS/cpsc330-2024W2) 

**Due date: [Feb 03, 11:59 pm](https://github.com/UBC-CS/cpsc330-2024W2?tab=readme-ov-file#deliverable-due-dates-tentative).**

## Imports 

In [ ]:
from hashlib import sha1
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

<div class="alert alert-info">

## Submission instructions
<hr>
rubric={points:5}

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2. 
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


Follow the [homework submission instructions](https://github.com/UBC-CS/cpsc330-2024W2/blob/master/docs/homework_instructions.md). 

1. Before submitting the assignment, run all cells in your notebook to make sure there are no errors by doing `Kernel -> Restart Kernel and Clear All Outputs` and then `Run -> Run All Cells`. 
2. Notebooks with cell execution numbers out of order or not starting from "1" will have marks deducted. Notebooks without the output displayed may not be graded at all (because we need to see the output in order to grade your work).
3. Follow the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2024W2/docs/homework_instructions.html), which include information on how to do your assignment and how to submit your assignment.
4. Upload your solution on Gradescope. Check out this [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/) if you need help with Gradescope submission. 
5. Make sure that the plots and output are rendered properly in your submitted file. If the .ipynb file is too big and doesn't render on Gradescope, also upload a pdf or html in addition to the .ipynb so that the TAs can view your submission on Gradescope.


_Note: The assignments will get gradually more open-ended as we progress through the course. In many cases, there won't be a single correct solution. Sometimes you will have to make your own choices and your own decisions (for example, on what parameter values to use when they are not explicitly provided in the instructions). Use your own judgment in such cases and justify your choices, if necessary._

</div>

## Introduction <a name="in"></a>
<hr>

A crucial step when using machine learning algorithms on real-world datasets is preprocessing. This homework will give you some practice of data preprocessing and building a supervised machine learning pipeline on a real-world dataset. 

## Exercise 1: Dataset and preliminary EDA
<hr>


In this homework, you will be working on [the adult census dataset](https://www.kaggle.com/uciml/adult-census-income#). Download the CSV and save it as `adult.csv` under the data folder in this homework folder. 

This is a classification dataset and the classification task is to predict whether income exceeds 50K per year or not based on the census data. You can find more information on the dataset and features [here](http://archive.ics.uci.edu/ml/datasets/Adult).

The starter code below loads the data CSV (assuming that it is saved as `adult.csv` under the data folder). 

_Note that many popular datasets have sex as a feature where the possible values are male and female. This representation reflects how the data were collected and is not meant to imply that, for example, gender is binary._

In [ ]:
census_df = pd.read_csv("data/adult.csv")
census_df.shape

<br><br>

### 1.1 Data splitting 
rubric={autograde}

In order to avoid violation of the golden rule, the first step before we do anything is splitting the data. 

**Your tasks:**

1. Split the data into `train_df` (40%) and `test_df` (60%) with `random_state = 123`. Keep the target column (`income`) in the splits so that we can use it in the exploratory data analysis.  

_Typically, having more data for training is beneficial. However, in this lab, we'll be using a 40%/60% split because running cross-validation with this dataset can take a long time on a modest laptop. Using a smaller training set will make the model training a bit faster on your laptop. Additionally, with a larger test split, we?ll obtain a more reliable estimate of the model's performance!_

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 1

In [ ]:
train_df, test_df = train_test_split(census_df, random_state = 123, test_size=0.6)

...

In [ ]:
grader.check("q1.1")

<br><br>

Let's examine our `train_df`. 

In [ ]:
train_df.sort_index()

We see some missing values represented with a "?". Probably these were the questions not answered by some people during the census.  Usually `.describe()` or `.info()` methods would give you information on missing values. But here, they won't pick "?" as missing values because they are encoded as strings instead of an actual NaN in Python. So let's replace them with `np.nan` before we carry out EDA. If you do not do it, you'll encounter an error later on when you try to pass this data to a classifier. 

In [ ]:
train_df = train_df.replace("?", np.nan)
test_df = test_df.replace("?", np.nan)
train_df.shape

In [ ]:
train_df.sort_index()

The "?" symbols are now replaced with NaN values. 

<br><br>

### 1.2 `describe()` method
rubric={autograde}

**Your tasks:**

1. Examine the output of `train_df.describe()` with `include='all'` argument and store it in a variable called `census_summary`.
2. What are the highest hours per week someone reported? Store it in a variable called `max_hours_per_week`.
3. What is the most frequently occurring occupation in this dataset? Store it in a variable called `most_freq_occupation`.
4. Store the column names of the columns with missing values as a list in a variable called `missing_vals_cols`. 
5. Store the column names of all numeric-looking columns, irrespective of whether you want to include them in your model or not, as a list in a variable called `numeric_cols`.  

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 5

In [ ]:
census_summary = train_df.describe(include='all')
census_summary

In [ ]:
max_hours_per_week = 99.000000	

In [ ]:
most_freq_occupation = train_df["occupation"].mode()[0]

In [ ]:
...

In [ ]:
missing_vals_cols = train_df.columns[train_df.isnull().any()].tolist()
numeric_cols = train_df.select_dtypes(include=['number']).columns.tolist()

In [ ]:
# Sorting the lists for the autograder
missing_vals_cols.sort()
numeric_cols.sort()

In [ ]:
grader.check("q1.2")

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Visualizing features
rubric={viz,reasoning}

**Your tasks:**

1. For each numeric feature listed in `numeric_cols`, generate overlapping histograms for the <=50K and >50K classes, similar to how you did it in homework 2.
2. Provide a brief description of your preliminary observations based on the histograms. 

> You can use the library of your choice for visualization. 

<div class="alert alert-warning">

Solution_1.3
    
</div>

_Points:_ 4

In [ ]:
for feature in numeric_cols:
    train_df.groupby("income")[feature].plot.hist(bins=50, alpha=0.5, legend=True, density = True, title = "Histogram of " + feature + " grouped by income bracket");
    plt.xlabel(feature);
    plt.show()

2. From the histograms:

Age: Higher-income individuals (>50K) tend to be older. The normal distribution is shifted right for higher income, indicating older individuals are more likely to earn above 50K.
Education.num: There is a distinct separation. Higher education levels (education.num > 9) are more associated with >50K incomes.
Hours.per.week: Individuals working more hours per week are more likely to earn >50K, but there is still overlap.
Capital.loss and fnlwgt: These features are highly overlapping, meaning they do not provide clear information to distinguish income brackets.
Conclusion

The most informative numerical features for predicting income appear to be age, education.num, and hours.per.week. Other features like capital.loss and fnlwgt do not offer distinguishable patterns.
   

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Identifying different feature types and transformations  
<hr>

Typically, data isn't readily formatted for direct input into machine learning models. It's crucial for a machine learning practitioner to examine each column and determine an effective method for encoding its information. Let's determine the types of features we have and come up with suitable encoding strategies for them. 

<!-- BEGIN QUESTION -->

### 2.1 Identify transformations to apply
rubric={reasoning}

Before passing this data to a machine learning model, we need to apply some transformations on different features. Below we are providing possible transformations which can be applied on each column in `census_df`.  

**Your tasks:**
1. Write your justification or explanation for each row in the explanation column. An example explanation is given for the age column. 

> Note: This question is a bit open-ended. If you do not agree with the provided transformation, feel free to argue your case in the explanation. That said, in this assignment, go with the transformations provided below for the purpose of autograding. 

> You can find the information about the columns [here](http://archive.ics.uci.edu/ml/datasets/Adult).

<div class="alert alert-warning">

Solution_2.1
    
</div>

| Feature | Transformation | Explanation
| --- | ----------- | ----- |
| age | scaling with `StandardScaler` |  A numeric feature with no missing values, ranging from 17 to 90. Scaling is recommended due to its distinct range compared to other numeric features. While MinMaxScaler might be more suitable, using StandardScaler should be fine too.|
| workclass | imputation, one-hot encoding | |
| fnlwgt | drop |  |
| education | ordinal encoding | |
| education.num | drop | |
| marital.status | one-hot encoding  | |
| occupation | imputation, one-hot encoding  | |
| relationship | one-hot encoding  | |
| race | drop  |  |
| sex | one-hot encoding with "binary=True" | |
| capital.gain | scaling with `StandardScaler` |  | 
| capital.loss | scaling with `StandardScaler` |  |
| hours.per.week | scaling with `StandardScaler` | |
| native.country | imputation, one-hot encoding | | 


_Points:_ 13

| Feature | Transformation | Explanation
| --- | ----------- | ----- |
| age | scaling with `StandardScaler` |  A numeric feature with no missing values, ranging from 17 to 90. Scaling is recommended due to its distinct range compared to other numeric features. While MinMaxScaler might be more suitable, using StandardScaler should be fine too.|
| workclass | imputation, one-hot encoding | A categorical feature with 8 categories. There is no underlying ordinal order to the values, and an adult can only be assigned one workclass, so one-hot encoding is suitable. There are missing values so we need to impute. |
| fnlwgt | drop | This feature gives no information about the labels as seen in the histogram, seems like a randomly assigned identification number that would not be useful for predicting income. |
| education | ordinal encoding |A categorical fearure with no missing values, since there is an increased value associated with a greater amount education recieved, classes should be ordered ascending as follows: Preschool, 1st-4th, 5th-6th,  7th-8th, 9th, 10th, 11th, 12th, HS-grad, Assoc-voc,  Assoc-acdm, Some-college, Bachelors, Masters,  Doctorate .|
| education.num | drop | Repetitive of the above feature, education. Offers no new information |
| marital.status | one-hot encoding  |  A categorical feature with 7 categories: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse. There is no underlying ordinal order to the values,  and an adult can only be assigned one marital status, so one-hot encoding is suitable. There are no missing values. |
| occupation | imputation, one-hot encoding  | A categorical feature with 14 categories: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces. There is no underlying order to the values,  and an adult can only be assigned one occupation, so one-hot encoding is suitable. There are missing values so imputation is required. |
| relationship | one-hot encoding  | A categorical feature with 6 categories: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried. There is no underlying ordinal order to the values, and an adult can only be assigned one relationship status, so one-hot encoding is suitable. There are no missing values. |
| race | drop  | May lead to biased classification, due to confounding with other variables. |
| sex | one-hot encoding with "binary=True" | A binary categorical feature: Female or Male. No missing values. |
| capital.gain | scaling with `StandardScaler` | A skewed numerical feature ranging from 0 to 99999, with most individuals reporting zero gains and a few reporting very large values. Standardizing ensures that extreme values do not dominate the model, preventing them from disproportionately influencing predictions by normalizing the data.  | 
| capital.loss | scaling with `StandardScaler` | A skewed numerical feature ranging from 0 to 4356, with many zero values and occasional large losses. Standardizing this feature by normalizing the it ensures that models treat it proportionally rather than being biased by its raw scale. |
| hours.per.week | scaling with `StandardScaler` | A numerical feature ranging from 1 to 99. The feature generally follows a gaussian distribution and scaling is recommended due to its distinct range compared to other numeric features. |
| native.country | imputation, one-hot encoding | A categorical feature with 40 categories. There is no underlying ordinal order to the values, and an adult can only be assigned one native country, so one-hot encoding is suitable. There are missing values so we need to impute.| 


_Points:_ 13

<!-- END QUESTION -->

<br><br>

### 2.2 Identify feature types 
rubric={autograde}


**Your tasks:**
1. Based on the types of transformations we want to apply on the features above, identify different feature types and store them in the variables below as lists.  

<div class="alert alert-warning">
    
Solution_2.2
    
</div>

_Points:_ 5

In [ ]:
# Fill in the lists below.
numeric_features = ["age", "capital.gain", "capital.loss", "hours.per.week"]
categorical_features = ["workclass", "marital.status", "occupation", "relationship", "native.country"]
ordinal_features = ["education"]
binary_features = ["sex"]
drop_features = ["fnlwgt", "education.num", "race"]

target = "income"

...

In [ ]:
numeric_features.sort()
categorical_features.sort()
ordinal_features.sort()
binary_features.sort()
drop_features.sort()

In [ ]:
grader.check("q2.2")

<br><br><br><br>

## Exercise 3: Baseline models 

### 3.1 Separating feature vectors and targets  
rubric={autograde}

**Your tasks:**

1. Create `X_train`, `y_train`, `X_test`, `y_test` from `train_df` and `test_df`. 

<div class="alert alert-warning">
    
Solution_3.1
    
</div>

_Points:_ 1

In [ ]:
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_test, y_test = test_df.drop(columns=[target]), test_df[target]

In [ ]:
grader.check("q3.1")

<br><br>

### 3.2 Dummy classifier
rubric={autograde}

**Your tasks:**

1. Carry out 5-fold cross-validation using [`scikit-learn`'s `cross_validate`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) function with `return_train_score=True` and store the results as a dataframe named `dummy_df` where each row corresponds to the results from a cross-validation fold. 

<div class="alert alert-warning">
    
Solution_3.2
    
</div>

_Points:_ 1

In [ ]:
model = DummyClassifier(strategy="most_frequent")

cv_results = cross_validate(model, X_train, y_train, cv=5, return_train_score=True)

dummy_df = pd.DataFrame(cv_results)

In [ ]:
grader.check("q3.2")

<br><br>

<!-- BEGIN QUESTION -->

### 3.3 Discussion
rubric={reasoning}

**Your tasks:**
1. Hopefully, you successfully run cross-validation with the dummy classifier in the previous question. Now, if you were to train the [`sklearn`'s `SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) model on `X_train` and `y_train`, would it work? Briefly explain your reasoning.

<div class="alert alert-warning">
    
Solution_3.3
    
</div>

_Points:_ 2

It will not work because SVC expects numerical input. The categorical features have not yet been transformed and are still encoded as strings, the model will throw an error when trying to fit the data. Additionally, there are NaN missing values which we'd need to specify a strategy for SVC to handle. Finally, SVMs are sensitive to the scale of numerical features. Numeric features (e.g., age, hours.per.week) are not standardized SVC will perform poorly.

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 4: Column transformer 
<hr>

In this dataset, we have different types of features: numeric features, an ordinal feature, categorical features, and a binary feature. We want to apply different transformations on different columns and therefore we need a column transformer. First, we'll define different transformations on different types of features and then will create a `scikit-learn`'s `ColumnTransformer` using `make_column_transformer`. For example, the code below creates a `numeric_transformer` for numeric features. 

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_transformer = StandardScaler()

In the exercises below, you'll create transformers for other types of features. 

<br><br>

### 4.1 Preprocessing ordinal features
rubric={autograde}

**Your tasks:**

1. Create a transformer called `ordinal_transformer` for our ordinal features. 

> Ordering of some of the education levels is not obvious. Assume that "HS-grad" < "Prof-school" < "Assoc-voc" < "Assoc-acdm" < "Some-college" < "Bachelors"

<div class="alert alert-warning">
    
Solution_4.1
    
</div>

_Points:_ 5

In [ ]:
education_order = [
    'Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th', 
    'HS-grad', 'Prof-school', 'Assoc-voc', 'Assoc-acdm', 'Some-college', 
    'Bachelors', 'Masters', 'Doctorate'
]

# Initialize the OrdinalEncoder with the specified categories
ordinal_transformer = OrdinalEncoder(categories=[education_order], dtype=int)
...

In [ ]:
...

In [ ]:
...

In [ ]:
grader.check("q4.1")

<br><br>

### 4.2 Preprocessing binary features
rubric={autograde}

**Your tasks:**

1. Create a transformer called `binary_transformer` for our binary features to encode binary features as integers 0 and 1.

> _Note that many popular datasets have sex as a feature where the possible values are male and female. This representation reflects how the data were collected and is not meant to imply that, for example, gender is binary._

<div class="alert alert-warning">
    
Solution_4.2
    
</div>

_Points:_ 2

In [ ]:
binary_transformer = OneHotEncoder(drop="if_binary", dtype=int)
...

In [ ]:
grader.check("q4.2")

<br><br>

### 4.3 Preprocessing categorical features
rubric={autograde}

There are a few categorical features with missing values in our dataset. Our initial step is to impute these missing values before proceeding to one-hot encode the features. For this assignment, apply imputation to all categorical features, regardless of whether they have missing values. If a feature lacks missing values, the imputation step will have no effect.

If we want to apply more than one transformation on a set of features, we need to create a [`scikit-learn` `Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html). For example, for categorical features we can create a `scikit-learn` `Pipeline` with first step as imputation and the second step as one-hot encoding. 

**Your tasks:**

1. Create a `sklearn` `Pipeline` using [`make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) called `categorical_transformer` for our categorical features with two steps:
- `SimpleImputer` for imputation with `strategy="constant"` and `fill_value="missing"`
- `OneHotEncoder` with `handle_unknown="ignore"` and `sparse_output=False` for one-hot encoding. 

<div class="alert alert-warning">
    
Solution_4.3
    
</div>

_Points:_ 4

In [ ]:
categorical_transformer = make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="ignore", sparse_output=False))

In [ ]:
grader.check("q4.3")

<br><br>

### 4.4 Creating a column transformer. 
rubric={autograde}

**Your tasks:**
1. Create a `sklearn` `ColumnTransformer` named `preprocessor` using [`make_column_transformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.make_column_transformer.html) with the transformers defined in the previous exercises. Use the sequence below in the column transformer and add a "drop" step for the `drop_features` in the end.  
    - `numeric_transformer`
    - `ordinal_transformer`
    - `binary_transformer`
    - `categorical_transformer`
2. Transform the data by calling `fit_transform` on the training set and save it as a dataframe in a variable called `transformed_df`. How many new columns have been created in the preprocessed data in comparison to the original `X_train`? Store the difference between the number of columns in `transformed_df` and `X_train` in a variable called `n_new_cols`. 

> You are not required to do this but optionally you can try to get column names of the transformed data and create the dataframe `transformed_df` with proper column names. 

<div class="alert alert-warning">
    
Solution_4.4
    
</div>

_Points:_ 7

In [ ]:
preprocessor = make_column_transformer(
    (
        numeric_transformer, #scaling
        numeric_features, #the features on which to apply that transformer
    ),  # scaling on numeric features
    (
        ordinal_transformer, #ordinary encoding
        ordinal_features,
    ),  # Ordinal encoding on ordinal features
    (
        binary_transformer, #one hot encoding
        binary_features,
    ),  # OHE on categorical features
    (
        categorical_transformer, #categorical one hot encoding
        categorical_features,
    ),  # OHE on categorical features
    ("drop", drop_features),  #just drops these ones
)

In [ ]:
transformed_df = preprocessor.fit_transform(X_train) #the fit_transform function, first learns parameters like mean and sd, then applies them to the dataset
transformed_df = pd.DataFrame(transformed_df)

n_new_cols = transformed_df.shape[1] - X_train.shape[1]

categorical_transformer = preprocessor.transformers_[3][1]  # extracts the actual tranformer in third index aka categorical_transformer 

categorical_column_names = categorical_transformer.get_feature_names_out(categorical_features).tolist() #gets the new feature names created by the transformer, so returns the new column names

column_names = (
    numeric_features
    + ordinal_features
    + binary_features    
    + categorical_column_names
)

transformed_df.columns = column_names

transformed_df.head()

In [ ]:
grader.check("q4.4")

<br><br>

<!-- BEGIN QUESTION -->

### 4.5 Short answer questions
rubric={reasoning}

**Your tasks:**

Answer each of the following questions in 2 to 3 sentences. 

1. What is the problem with calling `fit_transform` on your test data with `StandardScaler`?
2. Why is it important to follow the Golden Rule? If you violate it, will that give you a worse classifier?
3. What are two advantages of using sklearn Pipelines? 
4. When is it appropriate to use sklearn `ColumnTransformer`? 

<div class="alert alert-warning">
    
Solution_4.5
    
</div>

_Points:_ 8

1. We violate the golden rule. StandardScaler computes the mean and standard deviation from the data when fit_transform is called. If you fit it  on the test data, the scaling will be based on statistics (mean, std) from the test rather than solely the training data.
2. If we let test data influence model training, our classifier that may overfit to trends in the training data, but generalize poorly to new data. The model is worse in the sense that it may learn patterns that do not generalize to new data, leading to artificially high validation scores but poor real-world performance. 
3. Using a Pipeline takes care of applying the fit_transform on the train portion and only transform on the validation portion in each fold, reducing change of data leakage. Pipelines streamline the machine learning workflow by bundling multiple preprocessing steps into a single object, which can be reused across models.
4. ColumnTransformer is useful when different preprocessing steps need to be applied to different feature types in a dataset, such as scaling numerical features, encoding categorical variables as one hot or ordinal, and imputing missing values. IT is used to build all our transformations together into one object, and that way we?re sure we do the same column transforms to all splits of the data.


<!-- END QUESTION -->

<br><br><br><br>

## Exercise 5: Building models 

Now that we have preprocessed features, we are ready to build models. Below, I'm providing the function we used in class which returns mean cross-validation score along with standard deviation for a given model. Use it to keep track of your results. 

In [ ]:
results_dict = {}  # dictionary to store all the results

In [ ]:
def mean_std_cross_val_scores(model, X_train, y_train, **kwargs):
    """
    Returns mean and std of cross validation

    Parameters
    ----------
    model :
        scikit-learn model
    X_train : numpy array or pandas DataFrame
        X in the training data
    y_train :
        y in the training data

    Returns
    ----------
        pandas Series with mean scores from cross_validation
    """

    scores = cross_validate(model, X_train, y_train, **kwargs)

    mean_scores = pd.DataFrame(scores).mean()
    std_scores = pd.DataFrame(scores).std()
    out_col = []

    for i in range(len(mean_scores)):
        out_col.append((f"%0.3f (+/- %0.3f)" % (mean_scores.iloc[i], std_scores.iloc[i])))

    return pd.Series(data=out_col, index=mean_scores.index)

Below, I'm showing an example where I call `mean_std_cross_val_scores` with `DummyClassifier`. The function calls `cross_validate` with the passed arguments and returns a series with mean cross-validation results and std of cross-validation. When you train new models, you can just add the results of these models in `results_dict`, which can be easily converted to a dataframe so that you can have a table with all your results. 

In [ ]:
# Baseline model

from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(random_state = 123)
pipe = make_pipeline(preprocessor, dummy)
results_dict["dummy"] = mean_std_cross_val_scores(
    pipe, X_train, y_train, cv=5, return_train_score=True
)
results_df = pd.DataFrame(results_dict).T
results_df

<br><br>

<!-- BEGIN QUESTION -->

### 5.1 Trying different classifiers
rubric={accuracy,quality}

**Your tasks:**

1. For each model provided in the starter code below:
    - Create a pipeline using `make_pipeline` with two steps: the preprocessor from section 4.4 and the model as your classifier.
    - Conduct 5-fold cross-validation using the pipeline. Obtain the mean cross-validation scores and standard deviation using the `mean_std_cross_val_scores` function provided earlier.
    - Record the results in a dataframe named `income_pred_results_df`. Use the model names from the models dictionary below as the index. Each row should represent results from the `mean_std_cross_val_scores` function. In essence, `income_pred_results_df` should resemble the `results_df` dataframe mentioned earlier, but with additional rows for the subsequent models.

> Note: The execution might take some time. Please be patient!"

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

models = {
    "decision tree": DecisionTreeClassifier(random_state=123),
    "kNN": KNeighborsClassifier(),
    "RBF SVM": SVC(random_state=123),
}

results = {}

for model_name, model in models.items():
    pipeline = make_pipeline(preprocessor, model)
    scores = mean_std_cross_val_scores(pipeline, X_train, y_train, cv=5, return_train_score=True )
    results[model_name] = scores
    
income_pred_results_df = pd.DataFrame.from_dict(results, orient="index")


<div class="alert alert-warning">
    
Solution_5.1
    
</div>

_Points:_ 10

In [ ]:
income_pred_results_df
...

In [ ]:
...

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 5.2 Discussion 
rubric={reasoning}

**Your tasks:**

Examine the train and validation accuracies and `fit` and `score` times for all the models in the results above and answer the questions below: 

1. Among the models, which one achieves the highest validation accuracy?
2. Excluding the dummy model, which model exhibits the shortest 'fit' time? Similarly, which model boasts the quickest 'score' time?
3. Among the models (excluding the dummy model), which one shows the highest degree of overfitting and which one exhibits the least overfitting?

<div class="alert alert-warning">
    
Solution_5.2
    
</div>

_Points:_ 5

1. The RBF SVM model achieved the highest validation accuracy.
2. The kNN model had the shortest fit time by a significant amount, however the decision tree had the shortest score time.
3. The decision tree shows the highest degree of overfitting as its training score is almost 1, and its test score is the lowest, indicating that the model is overfitting to the training dataset and struggles to generalize as the test score is much lower. The lowest degree of overfitting is shown by the SVM RBF as the test and train scores are practically identical, only differing by 0.003, which is within the margin of error as well.

<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 5.3 Hyperparameter optimization
rubric={accuracy,quality}

In this exercise, you'll carry out hyperparameter optimization for the hyperparameter `C` of SVC RBF classifier. In practice, you'll carry out hyperparameter optimization for all different hyperparameters of the most promising classifiers. For the purpose of this assignment, we'll only do it for the `SVC` classifier with one hyperparameter, namely `C`. 

**Your tasks:**

1. For each `C` value in the `param_grid` below: 
    - Create a pipeline object with two steps: preprocessor from 4.4 and `SVC` classifier with the `C` value.
    - Carry out 5-fold cross validation with the pipeline.  
    - Store the results in `results_dict` and display results as a pandas DataFrame. 
2. Which hyperparameter value seems to be performing the best? In this assignment, consider the hyperparameter value that gives you the highest cross-validation score as the "best" one. Store it in a variable called `best_C`. (Since this question is not autograded, please store the value directly as a number, something like `best_C = 0.001`, if `C = 0.001` is giving you the highest CV score.) Is it different than the default value for the hyperparameter used by `scikit-learn`? 

> Note: Running this will take a while. Please be patient. 

In [ ]:
param_grid = {"C": np.logspace(-1, 2, 4)}
param_grid
    

<div class="alert alert-warning">
    
Solution_5.3
    
</div>

_Points:_ 10

The best C value is 100.0, which is different that the default value used by sci-kit which is only 1.0

In [ ]:
results = {}
for c in param_grid["C"]:
    model = SVC(C=c, random_state=123)
    pipeline = make_pipeline(preprocessor, model)
    scores = mean_std_cross_val_scores(pipeline, X_train, y_train, cv=5, return_train_score=True )
    results[c]=scores

results_df = pd.DataFrame.from_dict(results, orient="index")

results_df

In [ ]:
results_df = pd.DataFrame.from_dict(results, orient="index")
results_df

In [ ]:
best_C = 100.0

...

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 6: Evaluating on the test set 
<hr>

Now that we have a best performing model, it's time to assess our model on the set aside test set. In this exercise, you'll examine whether the results you obtained using cross-validation on the train set are consistent with the results on the test set. 

### 6.1 Scoring on the unseen test set 
rubric={autograde}

**Your tasks:**

1. Create a pipeline named `final_pipeline` with the preprocessor from 4.4 as the first step and the best performing SVC model from 5.4 as the second step. 
2. Train the pipeline on the entire training set `X_train` and `y_train`. 
3. Score the pipeline on `X_test` and `y_test` and store the score in a variable called `test_score`.  

<div class="alert alert-warning">
    
Solution_6.1
    
</div>

_Points:_ 3

In [ ]:
final_pipeline = None
test_score = None

model = SVC(random_state=123, C=100)
final_pipeline=make_pipeline(preprocessor,model)
final_pipeline.fit(X_train,y_train)
test_score = final_pipeline.score(X_test,y_test)
...

In [ ]:
grader.check("q6.1")

<br><br><br><br>

## Exercise 7: Short answer questions

<!-- BEGIN QUESTION -->

### 7.1 kNNs toy example
rubric={points}

Suppose you want to get predictions using the $k$-nearest neighbour algorithm on the toy dataset below. 

$$ X = \begin{bmatrix}5 & 2\\4 & -2\\  2 & 2\\ 10 & 10\\ 9 & -1\\ 9& 9\end{bmatrix}, \quad y = \begin{bmatrix}0\\0\\1\\1\\1\\2\end{bmatrix}.$$

Suppose you are given a new data point $x=\begin{bmatrix} 0 & 0\end{bmatrix}$.  

1. What would the model predict when $k=1$?
2. What would the model predict when $k=3$?
3. What would the model predict when $k=3$ if we were doing regression rather than classification? 

<div class="alert alert-warning">
    
Solution_7.1
    
</div>

_Points:_ 4


First I'll calculate the distance of the point [0,0] to the other 6 points one by one
1. sqrt(5^2+2^2) = sqrt(29) 
2. sqrt(4^2+2^2) = sqrt(20)
3. sqrt(2^2+2^2) = sqrt(8)
4. sqrt(10^2+10^2) = sqrt(200)
5. sqrt(9^2+1^2) = sqrt(82)
6. sqrt(9^2+9^2) = sqrt(162)

For Q1, when k=1, the closest point to [0,0] is point 3 which is class 1, classifing x as 1
For Q2, the 3 the 3 closest points are points 1,2 and 3, which 2 of are class 0, classifying x as 0
For Q3, when using regression for k=3, the 3 closest points will be averaged which in this case is 0+0+1/3 = 0.333

<!-- END QUESTION -->

<br><br>

**Before submitting your assignment, please make sure you have followed all the instructions in the Submission instructions section at the top.** 



Congratulations on finishing the homework! This was a tricky one but I hope you are feeling good after working on it. You are now ready to build a simple supervised machine learning pipeline on real-world datasets! Well done :clap:! 

![](img/eva-well-done.png)

